# reparameterization-trick — worked example 3: Reparameterize with Varying Sigma Across the Batch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reparameterization-trick`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

In a VAE encoder, different examples in a batch typically produce different `(mu, logsigma)` outputs — each example gets its own posterior distribution over the latent space. The reparameterization trick handles this naturally: `t.randn_like(mu)` draws independent noise per batch element, and the `mu + sigma * eps` formula applies elementwise. Wider distributions (larger `sigma`) produce more spread samples.

## Worked solution

**Step 1 — construct batch-varying `mu` and `logsigma`.** We create a batch of 3 examples with different means and log-sigmas to make the varying-spread behavior visible.

**Step 2 — compute `sigma` from `logsigma`.** `sigma = (0.5 * logsigma).exp()`. For `logsigma = 0`, `sigma = 1`. For `logsigma = 2`, `sigma = exp(1) ≈ 2.72`.

**Step 3 — draw `eps` and compute `z`.** `eps = t.randn_like(mu)` gives one noise vector per batch element. `z = mu + sigma * eps` shifts and scales each independently.

**Step 4 — verify output statistics.** Over many samples, the per-element mean should approach `mu` and per-element std should approach `sigma`.

**Step 5 — print sigma values alongside sample spread.** This makes the connection between `logsigma` encoding and actual distribution width concrete.

In [ ]:
import torch as t

def reparameterize(mu: t.Tensor, logsigma: t.Tensor) -> t.Tensor:
    sigma = (0.5 * logsigma).exp()
    eps = t.randn_like(mu)
    return mu + sigma * eps

# --- exercise and print ---
t.manual_seed(0)

# 3 examples, 2 latent dims; each row has different mean and variance
mu       = t.tensor([[0.0, 0.0], [2.0, -2.0], [0.0, 0.0]])
logsigma = t.tensor([[0.0, 0.0], [0.0,  0.0], [2.0,  2.0]])
sigma    = (0.5 * logsigma).exp()

print('mu:     ', mu.tolist())
print('sigma:  ', sigma.tolist())

# Draw many samples to estimate empirical stats
N = 5000
all_z = t.stack([reparameterize(mu, logsigma) for _ in range(N)])  # (N, 3, 2)
print('\nEmpirical mean (expect ≈ mu):')
print(all_z.mean(0).round(decimals=2))
print('Empirical std (expect ≈ sigma):')
print(all_z.std(0).round(decimals=2))